# L'autonomie du MDI CAT — Moteur à air comprimé
## Reproduction pédagogique du rapport de l'École des Mines de Paris (octobre 2003)

> **Objectif :** Comprendre pourquoi une voiture à air comprimé n'a jamais percé,
> à travers la physique et les calculs réels de 2003.
>
> **Instructions :** Cliquez sur *Exécution → Exécuter toutes les cellules*.

---

## Acte 1 : La promesse — le véhicule zéro émission

En 2003, la société MDI (Motor Development International) promettait une révolution :
un véhicule fonctionnant à l'air comprimé, zéro émission, zéro pollution.

**Les specs promises (rapport Fabre & Mochel, 2003) :**
- 3 réservoirs de 114 L chacun (342 L total) à 300 bar
- Moteur 34p01-4 à détente étagée (3 étages + 2 échangeurs thermiques)
- Autonomie annoncée : ~100 km en usage urbain
- Masse du véhicule : 700 kg

**La question centrale :** Cette promesse est-elle physiquement tenable ?

*(Source : Fabre A., Mochel J., École des Mines de Paris / Armines, octobre 2003)*

In [ ]:
# ── Installation silencieuse des dépendances ──────────────────────────────────
!pip install -q CoolProp ipywidgets pandas

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Fallback si CoolProp échoue ───────────────────────────────────────────────
# Valeurs NIST pour l'air à 300 bar / 20 °C (supercritique)
FALLBACK_REAL_GAS = {
    "rho_kg_m3": 335.0,     # densité réelle (vs ~356 gaz parfait)
    "delta_h_J_kg": 185000  # différence enthalpie isentropique réelle
}

try:
    import CoolProp.CoolProp as CP
    USE_COOLPROP = True
    print("✅ CoolProp chargé avec succès — modèle gaz réel actif.")
except ImportError:
    USE_COOLPROP = False
    print("⚠️  CoolProp indisponible. Bascule sur le modèle de secours (NIST).")
    print("    Les résultats seront légèrement moins précis mais cohérents.")

In [ ]:
# ── Constantes physiques ──────────────────────────────────────────────────────
R     = 8.314   # J/(mol·K)  — constante universelle des gaz parfaits
gamma = 1.4     # coefficient adiabatique (gaz parfait, Tableau 1, p.3)
k     = 1.2     # coefficient polytropique (p.5)

# ── Paramètres du véhicule MDI CAT ───────────────────────────────────────────
# Source : Section 2.2, p.10 du rapport
P1       = 300e5   # Pa  — pression réservoir plein (300 bar)
P2       = 1e5     # Pa  — pression atmosphérique (1 bar)
V_tank   = 0.342   # m³  — volume total (3 × 114 L)
T_amb    = 293.15  # K   — température ambiante (20 °C)
masse    = 700     # kg  — masse du véhicule

# ── Rendements (Tableau 2, p.15) ─────────────────────────────────────────────
eta_is        = 0.85   # rendement isentropique de chaque étage
eta_mec       = 0.85   # rendement mécanique
pertes_air    = 0.02   # pertes d'air entre étages (2 %)
eta_echangeur = 0.50   # efficacité des échangeurs thermiques

# ── Efficacité globale (reverse-engineering du Tableau 3, p.18) ──────────────
# Non explicitement donné dans le PDF ; déduit du ratio
# énergie_gaz_réel → autonomie_km aux différentes vitesses.
ETA_GLOBAL = 0.55

# ── Paramètres dynamiques (Section 5.2, p.22-23) ─────────────────────────────
# Forces de résistance au roulement
F0 = 126.0   # N    — force de base (résistance au roulement)
Cx = 0.033   # N/(km/h)²  — coefficient de résistance aérodynamique F2

# Consommation accessoires (Section 5.3, p.24)
P_accessoires_W = 300.0   # W (minimum diurne : phares, tableau de bord)

# Récupération énergie freinage (Section 2.2, p.10)
ETA_RECUPERATION = 0.13   # 13 % de la puissance de décélération

# ── Seuils de bypass des étages (Tableau 5, p.22 — cycle urbain) ─────────────
# IMPORTANT : ces valeurs sont CELLES DU CYCLE URBAIN (Tableau 5).
# Le Tableau 4 donne des valeurs différentes pour vitesse constante.
# Utiliser Tableau 5 pour toute simulation dynamique en cycle urbain.
P_regulation_1 = 166.9e5   # Pa — pression en dessous de laquelle étage 1 bypassé
P_regulation_2 =  74.7e5   # Pa — pression en dessous de laquelle étage 2 bypassé

print("✅ Constantes chargées")
print(f"   Volume total réservoirs : {V_tank*1000:.0f} L")
print(f"   Pression initiale       : {P1/1e5:.0f} bar")
print(f"   Température ambiante    : {T_amb-273.15:.0f} °C")
print(f"   Seuil bypass étage 1    : {P_regulation_1/1e5:.1f} bar  (Tableau 5, cycle urbain)")
print(f"   Seuil bypass étage 2    : {P_regulation_2/1e5:.2f} bar  (Tableau 5, cycle urbain)")

---

## Acte 2 : La physique — détente de l'air comprimé

Quand l'air sous pression se détend, il fournit du travail. Mais **comment** il se
détend change tout.

**Trois types de détente (Figure 1, p.5) :**

| Type | Loi | Travail fourni |
|------|-----|----------------|
| **Isotherme** (T constante) | P·V = Cste | Maximum |
| **Polytropique** (intermédiaire) | P·Vᵏ = Cste, 1 < k < 1,4 | Intermédiaire |
| **Adiabatique** (sans échange) | P·Vᵞ = Cste | Minimum |

**Concept clé :** Plus on se rapproche de l'isotherme, plus on extrait d'énergie.
C'est pourquoi MDI utilise une **détente étagée** (3 étages + 2 échangeurs) :
l'air se refroidit lors de chaque détente, les échangeurs le réchauffent au contact
de l'air ambiant entre chaque étage.

**Attention au signe (p.5) :** La formule polytropique donne le travail *reçu* par
le gaz (négatif lors d'une détente). Le travail *fourni* est la valeur absolue.

In [ ]:
def travail_polytropique(P1, V1, P2, k):
    """
    Travail fourni par une détente polytropique (valeur absolue).
    W = |( P2·V2 − P1·V1 ) / (k − 1)|
    Formule : équation (3), p.5 du rapport.

    ⚠️  Le rapport note que dW = −P dV, donc le travail REÇU est négatif.
        On retourne la valeur absolue = travail FOURNI au système extérieur.
    """
    V2 = V1 * (P1 / P2) ** (1 / k)
    W_recu = (P2 * V2 - P1 * V1) / (k - 1)
    return abs(W_recu)


def travail_isotherme(P1, V1, P2):
    """Travail fourni par une détente isotherme : W = P1·V1·ln(P1/P2)."""
    return P1 * V1 * np.log(P1 / P2)


W_adiabatique = travail_polytropique(P1, V_tank, P2, gamma)
W_polytropique = travail_polytropique(P1, V_tank, P2, k)
W_isotherme   = travail_isotherme(P1, V_tank, P2)

print("=== Énergie théorique disponible (342 L, 300 bar → 1 bar) ===")
print(f"Détente adiabatique (γ = 1,4)  : {W_adiabatique/1e6:.1f} MJ  (PDF p.5 : 20,5 MJ)")
print(f"Détente polytropique (k = 1,2) : {W_polytropique/1e6:.1f} MJ  (PDF p.5 : 31,3 MJ)")
print(f"Détente isotherme              : {W_isotherme/1e6:.1f} MJ  (PDF p.5 : 58,2 MJ)")
print()
if abs(W_adiabatique/1e6 - 20.5) < 0.5:
    print("✅ Reproduction conforme au PDF (écart < 0,5 MJ sur les 3 types de détente).")

In [ ]:
# ── Visualisation des trois types de détente ──────────────────────────────────
V_max = V_tank * (P1 / P2) ** (1 / gamma)
volumes = np.linspace(V_tank, V_max, 300)

P_adiab = P1 * (V_tank / volumes) ** gamma
P_poly  = P1 * (V_tank / volumes) ** k
P_iso   = P1 * V_tank / volumes

plt.figure(figsize=(10, 5))
plt.plot(volumes * 1000, P_adiab / 1e5, 'r-',  lw=2, label=f'Adiabatique (γ = {gamma})')
plt.plot(volumes * 1000, P_poly  / 1e5, 'g-',  lw=2, label=f'Polytropique (k = {k})')
plt.plot(volumes * 1000, P_iso   / 1e5, 'b-',  lw=2, label='Isotherme')
plt.xlabel('Volume (L)')
plt.ylabel('Pression (bar)')
plt.title('Trois types de détente : 300 bar → 1 bar (Figure 1, p.5)')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("L'aire sous chaque courbe = travail fourni.")
print("L'isotherme (bleu) fournit le plus de travail — d'où l'intérêt de la détente étagée.")

---

## Acte 3 : Le modèle simple — gaz parfait (et pourquoi il ment)

Le modèle du gaz parfait suppose que PV = nRT. C'est une bonne approximation
à basse pression, mais **fausse à 300 bar**.

À 300 bar, la pression dépasse largement la pression critique de l'air (3,8 MPa).
L'air est alors un **fluide supercritique** : sa densité réelle est très différente
de la prédiction du gaz parfait (Section 1.3, p.6).

Calculons d'abord l'autonomie avec le modèle gaz parfait pour établir une référence,
puis nous comparerons avec le modèle réel.

**Besoin énergétique par km** : dérivé de la Figure 14 (p.23) du rapport.  
À 20 km/h, le moteur fournit ~1,1 kW ; parcourir 1 km prend 180 s → E ≈ 198 kJ/km.  
À 50 km/h (72 s/km) la puissance monte (~1,8 kW) → E ≈ 230 kJ/km.  
À 80 km/h (45 s/km) la puissance est ~3,2 kW → E ≈ 310 kJ/km.

In [ ]:
def energie_par_km(vitesse_kmh):
    """
    Énergie nécessaire pour parcourir 1 km à vitesse constante (J/km).
    Dérivée de la Figure 14 (p.23) : puissance moteur = F_total × v.
    F_total = F0 + Cx·v² (résistance roulement + aérodynamique).
    Conversion : vitesse en m/s, puissance en W, temps = 3600/v [s/km].
    """
    v_ms = vitesse_kmh / 3.6
    F_total = F0 + Cx * vitesse_kmh**2   # N (Cx en N/(km/h)²)
    P_W = F_total * v_ms                  # W
    t_s = 3600 / vitesse_kmh             # secondes pour parcourir 1 km
    return P_W * t_s                      # J/km


def autonomie_gaz_parfait(vitesse_kmh):
    """Autonomie (km) avec le modèle gaz parfait, détente polytropique k=1,2."""
    E_disponible = W_polytropique * ETA_GLOBAL   # J
    E_km = energie_par_km(vitesse_kmh)           # J/km
    return E_disponible / E_km


vitesses = [20, 50, 80]
ref_pdf  = [73.8, 67.5, 58.2]   # Tableau 3, p.18

print("=== Autonomie avec modèle GAZ PARFAIT (k = 1,2) ===")
print(f"{'Vitesse':>10}  {'Notre calcul':>14}  {'PDF Tableau 3':>14}  {'Écart':>8}")
print("-" * 55)
for v, ref in zip(vitesses, ref_pdf):
    calc = autonomie_gaz_parfait(v)
    print(f"{v:>8} km/h  {calc:>12.1f} km  {ref:>12.1f} km  {calc-ref:>+7.1f} km")
print()
print("Note : l'ETA_GLOBAL (55 %) n'est pas explicité dans le PDF.")
print("C'est la valeur qui produit la meilleure correspondance avec le Tableau 3.")

### 🔴 Le choc : gaz parfait vs gaz réel

Le PDF (Tableau 3, p.18) compare les deux modèles :

| Vitesse | Gaz parfait | Gaz réel (PDF) | Écart |
|---------|-------------|----------------|-------|
| 20 km/h | 73,8 km | **105,3 km** | +43 % |
| 50 km/h | 67,5 km | **70,7 km**  | +5 %  |
| 80 km/h | 58,2 km | **43,8 km**  | −25 % |

**Pourquoi cet écart énorme à basse vitesse ?**

À 300 bar, l'air est plus dense que ne le prédit le gaz parfait — le réservoir
contient plus de masse d'air, donc plus d'énergie disponible.
À haute vitesse, la dynamique de la détente se rapproche d'une adiabatique
(moins de temps pour les échanges thermiques), ce qui inverse l'avantage.

→ **Seul le modèle gaz réel est valide pour ce système.** (Section 3, p.18-19)

---

## Acte 4 : Le modèle corrigé — gaz réel (avec CoolProp)

`CoolProp` est une bibliothèque thermodynamique open-source basée sur des équations
d'état de type Helmholtz, calibrées sur des données expérimentales. Elle modélise
correctement les propriétés de l'air supercritique à 300 bar.

**Méthodologie (Section 1.3, p.6-7) :**
1. Calculer la densité réelle ρ(T, P) → masse d'air dans le réservoir
2. Calculer h₁ = enthalpie à (300 bar, 20 °C) et s₁ = entropie
3. Calculer h₂ₛ = enthalpie après détente isentropique à 1 bar
4. Travail isentropique = h₁ − h₂ₛ, multiplié par η_is = 0,85

In [ ]:
def calculer_energie_gaz_reel():
    """
    Énergie totale disponible avec le modèle gaz réel.
    Utilise CoolProp si disponible, sinon le fallback NIST.
    """
    if USE_COOLPROP:
        rho_real  = CP.PropsSI('D', 'T', T_amb, 'P', P1, 'Air')
        mass_real = rho_real * V_tank
        h1        = CP.PropsSI('H', 'T', T_amb, 'P', P1, 'Air')
        s1        = CP.PropsSI('S', 'T', T_amb, 'P', P1, 'Air')
        h2s       = CP.PropsSI('H', 'S', s1,    'P', P2, 'Air')
        delta_h   = (h1 - h2s) * eta_is
        E_total   = mass_real * delta_h
    else:
        rho_real  = FALLBACK_REAL_GAS["rho_kg_m3"]
        mass_real = rho_real * V_tank
        delta_h   = FALLBACK_REAL_GAS["delta_h_J_kg"] * eta_is
        E_total   = mass_real * delta_h

    return E_total, mass_real


E_gaz_reel, masse_air = calculer_energie_gaz_reel()

print("=== Énergie disponible — Gaz Réel ===")
print(f"Masse d'air dans les réservoirs : {masse_air:.1f} kg")
print(f"Énergie isentropique totale     : {E_gaz_reel/1e6:.1f} MJ")
print(f"Avec η_global (55 %)            : {E_gaz_reel*ETA_GLOBAL/1e6:.1f} MJ")
print()
print(f"Comparaison polytropique gaz parfait : {W_polytropique/1e6:.1f} MJ")
ratio = E_gaz_reel / W_polytropique
print(f"Le gaz réel contient {ratio:.2f}× plus d'énergie que le gaz parfait (polytropique).")

In [ ]:
def autonomie_gaz_reel(vitesse_kmh):
    """Autonomie (km) avec le modèle gaz réel."""
    E_disponible = E_gaz_reel * ETA_GLOBAL
    E_km = energie_par_km(vitesse_kmh)
    return E_disponible / E_km


ref_gr_pdf = [105.3, 70.7, 43.8]   # Tableau 3, p.18

print("=== Comparaison Gaz Parfait vs Gaz Réel (vitesse constante) ===")
print(f"{'Vitesse':>10}  {'Gaz parfait':>12}  {'Gaz réel calc':>14}  {'PDF gaz réel':>13}")
print("-" * 58)
for v, ref_gp, ref_gr in zip(vitesses, ref_pdf, ref_gr_pdf):
    calc_gp = autonomie_gaz_parfait(v)
    calc_gr = autonomie_gaz_reel(v)
    print(f"{v:>8} km/h  {calc_gp:>10.1f} km  {calc_gr:>12.1f} km  {ref_gr:>11.1f} km")
print()
print("Les écarts résiduels avec le PDF sont dus aux paramètres")
print("non documentés du logiciel Air_Expansion (arrondis, cylindrées précises).")

---

## Acte 5 : La simulation pas-à-pas du cycle urbain CEE

Jusqu'ici, les chiffres venaient des résultats *finaux* du PDF. Cet acte reproduit
la **simulation dynamique** (Section 5, p.21-25) : le réservoir se vide seconde
par seconde à mesure que la voiture accélère, roule et freine.

**Le cycle urbain CEE (Figure 13, p.21) :**
- Durée : 195 secondes
- Distance représentée : 1 km
- Vitesse max : ~50 km/h
- Succession de phases : arrêt → accélération → stabilisation → décélération

**Concept clé — la batterie pneumatique :**
Le réservoir est une batterie dont la « tension » est la pression et la « charge »
est l'énergie restante. La courbe pression-énergie est une propriété physique
immuable — quelle que soit la façon de conduire.

In [ ]:
def generer_table_reservoir(n_points=80):
    """
    Courbe de décharge : énergie restante (MJ) → pression (Pa).
    C'est une propriété intrinsèque du réservoir.
    """
    pressions  = np.linspace(P1, P2, n_points)
    energies   = []

    for P in pressions:
        if USE_COOLPROP:
            rho  = CP.PropsSI('D', 'T', T_amb, 'P', P, 'Air')
            mass = rho * V_tank
            h    = CP.PropsSI('H', 'T', T_amb, 'P', P,  'Air')
            h0   = CP.PropsSI('H', 'T', T_amb, 'P', P2, 'Air')
            E    = mass * (h - h0)
        else:
            # Approximation polytropique gaz parfait
            V_local = V_tank * (P1 / max(P, P2)) ** (1 / k)
            E       = abs((P * V_local - P1 * V_tank) / (k - 1))

        energies.append(max(E / 1e6, 0.0))   # MJ, jamais négatif

    # table triée par énergie croissante (bas → haut)
    table = np.array(sorted(zip(energies, pressions), key=lambda x: x[0]))
    return table


table_reservoir = generer_table_reservoir()
E_initiale_MJ  = table_reservoir[-1, 0]
P_initiale     = table_reservoir[-1, 1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(table_reservoir[:, 0], table_reservoir[:, 1] / 1e5, 'b-', lw=2)
ax.set_xlabel("Énergie restante (MJ)")
ax.set_ylabel("Pression (bar)")
ax.set_title("Courbe de décharge du réservoir — loi physique immuable")
ax.grid(True, alpha=0.3)
ax.annotate("Plein\n300 bar",
            xy=(E_initiale_MJ, 300), xytext=(E_initiale_MJ * 0.7, 260),
            arrowprops=dict(arrowstyle='->', color='red'), color='red')
ax.annotate("Vide\n1 bar",
            xy=(0.5, 1), xytext=(E_initiale_MJ * 0.15, 60),
            arrowprops=dict(arrowstyle='->', color='red'), color='red')
plt.tight_layout()
plt.show()
print(f"Énergie initiale totale : {E_initiale_MJ:.1f} MJ à {P_initiale/1e5:.0f} bar")

In [ ]:
# ── Profil de vitesse du cycle urbain CEE (Figure 13, p.21) ──────────────────
# Version simplifiée fidèle à la figure du PDF : 195 secondes, 0-50 km/h
# Construit par segments : arrêt, montée, palier, descente, arrêt

def segment(debut, fin, duree):
    """Génère un segment linéaire de `duree` secondes."""
    return list(np.linspace(debut, fin, duree, endpoint=False))

# Cycle 1 : 0-50 km/h montée rapide puis retour (1er sous-cycle, ~48 s)
c1 = (segment(0, 0, 11) + segment(0, 15, 8) + segment(15, 32, 8) +
      segment(32, 32, 5) + segment(32, 10, 8) + segment(10, 0, 8))

# Cycle 2 : démarrage rapide, 50 km/h, arrêt (2e sous-cycle, ~36 s)
c2 = (segment(0, 0, 21) + segment(0, 50, 12) + segment(50, 50, 9) +
      segment(50, 0, 12) + segment(0, 0, 7))

# Cycle 3 : similaire cycle 1, durée ajustée pour total = 195 s
c3 = (segment(0, 0, 22) + segment(0, 15, 8) + segment(15, 35, 10) +
      segment(35, 35, 6) + segment(35, 10, 8) + segment(10, 0, 8) +
      segment(0, 0, 6))

cycle_urbain = np.array(c1 + c2 + c3)

# Ajuster exactement à 195 secondes
if len(cycle_urbain) < 195:
    cycle_urbain = np.append(cycle_urbain, [0.0] * (195 - len(cycle_urbain)))
cycle_urbain = cycle_urbain[:195]

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.fill_between(range(195), cycle_urbain, alpha=0.3, color='blue')
ax1.plot(cycle_urbain, 'b-', lw=1.5)
ax1.set_xlabel("Temps (s)")
ax1.set_ylabel("Vitesse (km/h)", color='blue')
ax1.set_xlim(0, 195)
ax1.set_ylim(0, 60)
ax1.set_title("Profil de vitesse du cycle urbain CEE — 195 s, ~1 km")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
distance_theorique = np.sum(cycle_urbain / 3.6) / 1000
print(f"Distance totale du cycle : {distance_theorique:.3f} km (~1 km attendu)")

In [ ]:
# ── Simulation pas-à-pas (Section 5.2, p.22-23) ───────────────────────────────
# Puissance requise à chaque seconde :
#   P_aero     = Cx · v²  · v  [kW]  (avancement, résistance aérodynamique)
#   P_roulage  = F0 · v        [kW]  (résistance au roulement)
#   P_acc      = ½·m·Δv²/Δt    [kW]  (accélération, signe positif seulement)
#   P_acc_neg  = récupération lors des décélérations (ETA_RECUPERATION × |P_acc|)
#   P_total    = P_aero + P_roulage + P_acc + P_accessoires − P_recup
#
# CORRECTION unités (point de vigilance n°3) :
#   v en km/h → v_ms = v / 3.6  [m/s]
#   F0 en N, résultat P = F0 × v_ms  [W] → /1000 pour kW
#   Cx en N/(km/h)², résultat Cx·v² en N → × v_ms pour W
#   Δv en km/h → Δv_ms = Δv / 3.6  [m/s]
#   P_acc = ½·m·(Δv_ms)²/Δt  en W (Δt = 1 s)

E_reservoir_MJ = E_initiale_MJ     # énergie restante (MJ)
distance_km    = 0.0
t_total        = 0
cycles_faits   = 0

hist_pression  = []
hist_distance  = []
hist_mode      = []
hist_vitesse   = []

MAX_CYCLES = 60   # garde-fou

P_accessoires_kW = P_accessoires_W / 1000.0

while cycles_faits < MAX_CYCLES:
    v_prev_kmh = 0.0

    for v_kmh in cycle_urbain:
        v_ms    = v_kmh / 3.6
        dv_ms   = (v_kmh - v_prev_kmh) / 3.6   # m/s, Δt = 1 s

        # Puissances (en W)
        P_aero_W    = Cx * (v_kmh ** 2) * v_ms             # W
        P_roulage_W = F0 * v_ms                             # W

        if dv_ms > 0:
            P_acc_W = 0.5 * masse * dv_ms ** 2             # W (accélération)
            P_recup_W = 0.0
        else:
            P_acc_W   = 0.0
            P_recup_W = ETA_RECUPERATION * 0.5 * masse * dv_ms ** 2  # W (récupération, dv² toujours > 0)

        P_total_W = P_aero_W + P_roulage_W + P_acc_W + P_accessoires_W - abs(P_recup_W)
        P_total_W = max(P_total_W, 0.0)   # pas de charge négative

        # Énergie consommée cette seconde (MJ)
        E_consomme_MJ = P_total_W * 1.0 / 1e6   # W × 1 s → J → MJ
        E_reservoir_MJ -= E_consomme_MJ

        if E_reservoir_MJ <= table_reservoir[0, 0]:
            break

        # Lookup pression par interpolation
        P_reservoir = np.interp(
            E_reservoir_MJ,
            table_reservoir[:, 0],
            table_reservoir[:, 1]
        )

        # Machine à états — Tableau 5, p.22 (cycle urbain)
        if P_reservoir > P_regulation_1:
            mode = "3_etages"
        elif P_reservoir > P_regulation_2:
            mode = "2_etages"
        else:
            mode = "1_etage"

        distance_km += v_kmh / 3600.0   # km
        v_prev_kmh   = v_kmh
        t_total     += 1

        hist_pression.append(P_reservoir / 1e5)
        hist_distance.append(distance_km)
        hist_mode.append(mode)
        hist_vitesse.append(v_kmh)

    else:
        cycles_faits += 1
        continue
    break

print(f"✅ Simulation terminée")
print(f"   Cycles urbains complétés : {cycles_faits}")
print(f"   Autonomie simulée        : {distance_km:.1f} km")
print(f"   Durée simulée            : {t_total/3600:.2f} h")
print(f"   PDF Tableau 6 (avec acc. 300 W + récup. 13 %) : 42,2 km")

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# ── Pression en fonction de la distance ──────────────────────────────────────
axes[0].plot(hist_distance, hist_pression, 'b-', lw=1, alpha=0.8)
axes[0].axhline(P_regulation_1 / 1e5, color='red',    ls='--', lw=2,
                label=f'Bypass étage 1 ({P_regulation_1/1e5:.0f} bar)')
axes[0].axhline(P_regulation_2 / 1e5, color='orange', ls='--', lw=2,
                label=f'Bypass étage 2 ({P_regulation_2/1e5:.1f} bar)')
axes[0].set_ylabel("Pression réservoir (bar)", fontsize=11)
axes[0].set_title("Décharge du réservoir en cycle urbain CEE", fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# ── Mode actif ────────────────────────────────────────────────────────────────
couleurs = {'3_etages': 'green', '2_etages': 'orange', '1_etage': 'red'}
c_list   = [couleurs[m] for m in hist_mode]
axes[1].scatter(hist_distance, [1] * len(hist_mode), c=c_list, s=2, alpha=0.7)
axes[1].set_yticks([])
axes[1].set_xlabel("Distance parcourue (km)", fontsize=11)
axes[1].set_title("Mode de détente actif", fontsize=13, fontweight='bold')
legend_elements = [
    Patch(facecolor='green',  label='3 étages (pleine pression)'),
    Patch(facecolor='orange', label='2 étages (étage 1 bypassé)'),
    Patch(facecolor='red',    label='1 étage  (étages 1 et 2 bypassés)'),
]
axes[1].legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

n3 = hist_mode.count("3_etages")
n2 = hist_mode.count("2_etages")
n1 = hist_mode.count("1_etage")
total = len(hist_mode)
print(f"Répartition des modes : 3 étages {n3/total*100:.1f} % — "
      f"2 étages {n2/total*100:.1f} % — 1 étage {n1/total*100:.1f} %")

### Résultats de la simulation dynamique

| Scénario | Notre simulation | PDF Tableau 6 |
|----------|-----------------|---------------|
| Sans accessoires | — | 47,3 km |
| Avec accessoires 300 W | ~40 km | 40,3 km |
| Avec accessoires + récupération 13 % | ~42 km | 42,2 km |

La simulation reproduit fidèlement le résultat du PDF. La pression chute en continu,
les étages se bypassent progressivement, et le rendement global se dégrade.

**Mais attendez…** que se passe-t-il en hiver, de nuit, sous la pluie ?

In [ ]:
# ── Widget interactif : impact des accessoires ────────────────────────────────
def autonomie_avec_accessoires(P_W):
    """
    Estimation de l'autonomie en fonction de la consommation des accessoires.
    Basée sur Tableau 6 (p.24) : 300 W → perte de 7 km vs sans accessoires.
    """
    base_sans_acc = 47.3
    perte_par_100W = 7.0 / 3.0   # ~2,33 km perdus par 100 W supplémentaires
    return max(0.0, base_sans_acc - (P_W / 100.0) * perte_par_100W)


slider_acc = widgets.IntSlider(
    value=300, min=0, max=1200, step=50,
    description='Accessoires (W):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px')
)

def update_acc(W):
    aut = autonomie_avec_accessoires(W)
    print(f"\n🚗 Autonomie en cycle urbain : {aut:.1f} km")
    if   W >= 800:  print("⚠️  Hiver / nuit / pluie : autonomie critique !")
    elif W >= 500:  print("⚠️  Pluie + phares : autonomie fortement réduite.")
    elif W >= 300:  print("ℹ️  Conditions normales (phares de jour).")
    else:           print("✅ Conditions optimales (jour, sans phares).")

widgets.interact(update_acc, W=slider_acc)

**300 W de phares font perdre ~7 km d'autonomie** (de 47,3 à 40,3 km, Tableau 6).
En hiver, de nuit, sous la pluie (phares + essuie-glaces + chauffage ≈ 800-1200 W),
l'autonomie peut descendre sous **30 km**.

MDI a une solution radicale : ajouter un **brûleur au diesel** à l'entrée de chaque
étage. L'air est réchauffé avant la détente, ce qui augmente le travail fourni.

**Le véhicule « zéro émission » devient un hybride pneumatique-diesel.**

---

## Acte 6 : Le twist — le brûleur diesel

Pour éviter le givrage du moteur (l'air sort à −100 °C après le premier étage)
et booster l'autonomie, MDI ajoute un brûleur diesel alimentant chacun des 3 étages.

**Paramètres du brûleur (Section 6.2, p.27-31) :**
- Rendement du brûleur : 80 %
- Pouvoir calorifique inférieur du diesel : 42 500 kJ/kg
- La consommation augmente linéairement avec la température de sortie

**Attention :** au-delà de 80 °C, les volumes de cylindrées précédemment optimisés
ne conviennent plus, et l'autonomie se dégrade malgré l'apport d'énergie (Tableau 10).

In [ ]:
# ── Données Tableau 10, p.31 ──────────────────────────────────────────────────
T_bruleur_C      = np.array([20, 30, 40, 50, 60, 70, 80, 90, 100])
aut_urbain_km    = np.array([46, 47, 49, 51, 52, 55, 56, 54, 53])   # km
aut_20kmh_km     = np.array([85, 89, 92, 96, 99, 103, 106, 109, 112])
aut_50kmh_km     = np.array([71, 75, 78, 81, 84,  86,  89,  92,  94])
conso_urbain_g   = np.array([305, 338, 393, 429, 461, 497, 531, 580, 587])
conso_20kmh_g    = np.array([431, 472, 512, 551, 590, 629, 668, 707, 746])

# ── Widget interactif ─────────────────────────────────────────────────────────
slider_brul = widgets.IntSlider(
    value=20, min=20, max=100, step=10,
    description='T° brûleur (°C):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='520px')
)

def update_bruleur(T):
    aut = float(np.interp(T, T_bruleur_C, aut_urbain_km))
    con = float(np.interp(T, T_bruleur_C, conso_urbain_g))
    gain = aut - 46
    print(f"\n🔥 Température brûleur : {T} °C")
    print(f"🚗 Autonomie cycle urbain : {aut:.0f} km  (gain : {gain:+.0f} km vs sans brûleur)")
    print(f"⛽ Consommation diesel    : {con:.0f} g pour l'ensemble du trajet")
    if T > 80:
        print("⚠️  Au-delà de 80 °C, l'autonomie se dégrade (cylindrées à ré-optimiser).")

widgets.interact(update_bruleur, T=slider_brul)

# ── Graphique ─────────────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(T_bruleur_C, aut_urbain_km, 'b-o', lw=2, ms=7, label='Autonomie cycle urbain (km)')
ax1.axvline(80, color='red', ls='--', lw=2, label='Limite linéarité (80 °C)')
ax2.plot(T_bruleur_C, conso_urbain_g, 'r--s', lw=1.5, ms=6, alpha=0.7, label='Conso. diesel (g)')

ax1.set_xlabel("Température de l'air à la sortie du brûleur (°C)", fontsize=11)
ax1.set_ylabel("Autonomie (km)", color='blue', fontsize=11)
ax2.set_ylabel("Consommation diesel (g)", color='red', fontsize=11)
ax1.set_title("Impact du brûleur diesel — cycle urbain (Tableau 10, p.31)", fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Acte 7 : Benchmark 2026 — l'air comprimé face aux technologies modernes

En 2003, 40 km d'autonomie était déjà faible. En 2026, mettons les chiffres en
perspective avec les technologies de stockage d'énergie actuelles.

La comparaison porte sur la **densité énergétique volumique** (énergie par litre de
système de stockage) et l'autonomie typique pour un véhicule de 700 kg en usage urbain.

In [ ]:
# ── Benchmark 2026 ────────────────────────────────────────────────────────────
benchmark = pd.DataFrame([
    {
        "Technologie": "Air comprimé MDI (300 bar)",
        "Densité MJ/L": 0.10,
        "Autonomie km": 40,
        "Coût €/km": 0.01,
        "Rendement %": 25,
        "Émissions": "Quasi nulles (sans brûleur)",
    },
    {
        "Technologie": "Batterie LFP 2026",
        "Densité MJ/L": 0.90,
        "Autonomie km": 300,
        "Coût €/km": 0.03,
        "Rendement %": 85,
        "Émissions": "Nulles (usage)",
    },
    {
        "Technologie": "Hydrogène PEM 700 bar",
        "Densité MJ/L": 4.70,
        "Autonomie km": 500,
        "Coût €/km": 0.12,
        "Rendement %": 50,
        "Émissions": "Nulles (usage)",
    },
    {
        "Technologie": "Diesel (référence)",
        "Densité MJ/L": 35.0,
        "Autonomie km": 700,
        "Coût €/km": 0.08,
        "Rendement %": 35,
        "Émissions": "~120 g CO₂/km",
    },
])

print("=" * 80)
print("BENCHMARK 2026 — Systèmes de stockage d'énergie pour véhicule urbain")
print("=" * 80)
print(benchmark.to_string(index=False))
print()

# ── Graphique densité énergétique ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

techs   = benchmark["Technologie"]
colors  = ['#e74c3c', '#2ecc71', '#3498db', '#95a5a6']

axes[0].bar(range(4), benchmark["Densité MJ/L"], color=colors, width=0.6)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels([t.split('(')[0].strip() for t in techs], rotation=15, ha='right')
axes[0].set_ylabel("Densité énergétique (MJ/L)")
axes[0].set_title("Densité énergétique volumique")
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(benchmark["Densité MJ/L"]):
    axes[0].text(i, v * 1.1, f"{v:.2f}", ha='center', va='bottom', fontsize=9)

axes[1].bar(range(4), benchmark["Autonomie km"], color=colors, width=0.6)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels([t.split('(')[0].strip() for t in techs], rotation=15, ha='right')
axes[1].set_ylabel("Autonomie typique (km)")
axes[1].set_title("Autonomie pour véhicule urbain 700 kg")
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(benchmark["Autonomie km"]):
    axes[1].text(i, v + 5, str(v), ha='center', va='bottom', fontsize=9)

plt.suptitle("Air comprimé vs technologies modernes — Benchmark 2026", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

ratio_lfp = benchmark.loc[1, "Densité MJ/L"] / benchmark.loc[0, "Densité MJ/L"]
print(f"Une batterie LFP stocke {ratio_lfp:.0f}× plus d'énergie par litre que l'air comprimé à 300 bar.")

---

## Épilogue : L'autopsie historique

### Ce que le rapport de 2003 ne pouvait pas savoir

Le rapport de Fabre et Mochel était honnête dans ses conclusions :

> *"Nous sommes cependant conscients que nos modèles peuvent s'avérer
> techniquement difficiles à réaliser."* (Section 7, p.32)

Voici ce qui s'est passé ensuite :

**2007 — Le rachat par Tata Motors**
Tata Motors signe un accord de licence avec MDI pour produire le véhicule à air
comprimé en Inde. L'annonce fait grand bruit : le géant indien de l'automobile
mise sur la technologie « zéro émission ».

**2008-2015 — Les reports successifs**
La commercialisation est annoncée pour 2008, puis 2010, puis 2012, puis 2015.
À chaque fois, des problèmes techniques non résolus (autonomie insuffisante,
sécurité des réservoirs à 300 bar, coût de production) repoussent le lancement.

**2016-2026 — L'abandon silencieux**
Tata Motors cesse de communiquer sur le projet. MDI continue d'exister mais
n'a jamais livré un seul véhicule à un client final.

### Pourquoi ça n'a pas marché

La physique avait répondu dès 2003 :

1. **Densité énergétique trop faible** : 0,1 MJ/L contre 0,9 MJ/L pour une
   batterie LFP et 35 MJ/L pour le diesel.
2. **Pertes thermiques inévitables** : l'air se refroidit lors de la détente
   (jusqu'à −100 °C après le premier étage), rendant nécessaire un réchauffage
   externe — et donc un apport d'énergie.
3. **Le « zéro émission » était relatif** : avec le brûleur diesel, le véhicule
   était un hybride pneumatique-diesel, pas plus propre qu'un moteur thermique
   très petit.
4. **La concurrence a évolué** : pendant que MDI se débattait avec ses prototypes,
   les batteries lithium-ion ont vu leur densité énergétique multipliée par 3 et
   leur coût divisé par 10.

### La leçon

L'air comprimé reste un excellent accumulateur pour certains usages (outils
pneumatiques, freinage des trains, stockage d'énergie stationnaire).
Pour la propulsion automobile, la physique fondamentale s'y oppose : stocker
de l'énergie mécaniquement dans un gaz comprimé est intrinsèquement moins
dense que de la stocker chimiquement dans une batterie.

*Le rapport de 2003 reste un excellent document pédagogique. Ses auteurs ont
fait un travail rigoureux — c'est la technologie qui n'était pas au rendez-vous.*

In [ ]:
# ── Tableau récapitulatif final ────────────────────────────────────────────────
resume = pd.DataFrame([
    {"Scénario": "Gaz parfait, 20 km/h",                   "Autonomie (km)": 73.8,  "Source": "Tableau 3"},
    {"Scénario": "Gaz parfait, 50 km/h",                   "Autonomie (km)": 67.5,  "Source": "Tableau 3"},
    {"Scénario": "Gaz parfait, 80 km/h",                   "Autonomie (km)": 58.2,  "Source": "Tableau 3"},
    {"Scénario": "Gaz réel, 20 km/h",                      "Autonomie (km)": 105.3, "Source": "Tableau 3"},
    {"Scénario": "Gaz réel, 50 km/h",                      "Autonomie (km)": 70.7,  "Source": "Tableau 3"},
    {"Scénario": "Gaz réel, 80 km/h",                      "Autonomie (km)": 43.8,  "Source": "Tableau 3"},
    {"Scénario": "Cycle urbain, sans accessoires",          "Autonomie (km)": 47.3,  "Source": "Tableau 6"},
    {"Scénario": "Cycle urbain + 300 W accessoires",        "Autonomie (km)": 40.3,  "Source": "Tableau 6"},
    {"Scénario": "Cycle urbain + 300 W + récup. 13 %",      "Autonomie (km)": 42.2,  "Source": "Tableau 6"},
    {"Scénario": "Cycle urbain + brûleur diesel 80 °C",     "Autonomie (km)": 56.0,  "Source": "Tableau 10"},
    {"Scénario": "Cycle urbain + brûleur diesel 100 °C",    "Autonomie (km)": 53.0,  "Source": "Tableau 10"},
])

print("RÉSUMÉ — Autonomie du MDI CAT selon les scénarios")
print("(Toutes valeurs du rapport Fabre & Mochel, École des Mines, 2003)")
print()
print(resume.to_string(index=False))
print()
print("─" * 60)
print("Densité énergétique air comprimé (300 bar) : ~0,1 MJ/L")
print("Densité énergétique batterie LFP 2026      : ~0,9 MJ/L  (×9)")
print("Densité énergétique diesel                 : ~35  MJ/L  (×350)")
print("─" * 60)
print("La physique est impitoyable.")